In [1]:
import numpy as np
import pandas as pd
import datetime as dt

import dask.dataframe as dd
from pathlib import Path
import re

import torch
from tqdm import tqdm
from torch import multiprocessing

In [4]:
import sys

# append the path of the
# parent directory
sys.path.append(f'..')
sys.path.append(f'../src/')
sys.path.append(f'../src/models/bat_call_detector/batdetect2/')

import batdt2_pipeline

In [5]:
def get_params_relevant_to_data_at_location_all_usable_files(cfg):
    data_params = dict()
    data_params['site'] = cfg['site']
    print(f"Searching for files from {cfg['site']} in {cfg['year']}")

    hard_drive_df = dd.read_csv(f'../output_dir/ubna_data_*_collected_audio_records.csv', dtype=str).compute()
    if 'Unnamed: 0' in hard_drive_df.columns:
        hard_drive_df.drop(columns='Unnamed: 0', inplace=True)
    hard_drive_df["datetime_UTC"] = pd.DatetimeIndex(hard_drive_df["datetime_UTC"])
    hard_drive_df.set_index("datetime_UTC", inplace=True)
    
    files_from_location = filter_df_with_location_all_usable(hard_drive_df, cfg)
    data_params['output_dir'] = cfg["output_dir"] / (data_params["site"].split()[0])
    print(f"Will save csv file to {data_params['output_dir']}")

    data_params['ref_audio_files'] = sorted(list(files_from_location["file_path"].apply(lambda x : Path(x)).values))
    file_status_cond = files_from_location["file_status"] == "Usable for detection"
    good_location_df = files_from_location.loc[file_status_cond]
    data_params['good_audio_files'] = sorted(list(good_location_df["file_path"].apply(lambda x : Path(x)).values))

    if data_params['good_audio_files'] == data_params['ref_audio_files']:
        print("All files from deployment session good!")
    else:
        print("Error files exist!")

    print(f"Will be looking at {len(data_params['good_audio_files'])} files from {data_params['site']}")

    return good_location_df, data_params


def filter_df_with_location_all_usable(ubna_data_df, cfg):
    site_name_cond = ubna_data_df["site_name"] == cfg['site']
    file_year_cond = ubna_data_df.index.year == (dt.datetime.strptime(cfg['year'], '%Y')).year
    file_june_cond = ubna_data_df.index.month == (dt.datetime.strptime('June', '%B')).month
    file_july_cond = ubna_data_df.index.month == (dt.datetime.strptime('July', '%B')).month
    file_august_cond = ubna_data_df.index.month == (dt.datetime.strptime('August', '%B')).month
    file_september_cond = ubna_data_df.index.month == (dt.datetime.strptime('September', '%B')).month
    file_october_cond = ubna_data_df.index.month == (dt.datetime.strptime('October', '%B')).month
    file_month_cond1 = np.logical_or(file_july_cond, file_august_cond)
    file_month_cond2 = np.logical_or(file_september_cond, file_october_cond)
    file_month_cond3 = np.logical_or(file_june_cond, file_month_cond1)
    file_month_cond = np.logical_or(file_month_cond3, file_month_cond2)

    file_error_cond = np.logical_and((ubna_data_df["file_duration"]!='File has no comment due to error!'), (ubna_data_df["file_duration"]!='File has no Audiomoth-related comment'))
    all_errors_cond = np.logical_and((ubna_data_df["file_duration"]!='Is empty!'), file_error_cond)
    file_date_cond = np.logical_and(file_year_cond, file_month_cond)

    filtered_location_df = ubna_data_df.loc[site_name_cond&file_date_cond&all_errors_cond].sort_index()
    filtered_location_nightly_df = filtered_location_df.between_time(cfg['recording_start'], cfg['recording_end'], inclusive="left")

    return filtered_location_nightly_df

In [6]:
all_file_durations = dict()

In [25]:
cfg = dict()
cfg["site"] = 'Fallen Tree'
cfg["year"] = '2022'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '23:59'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df_fallen, data_params = get_params_relevant_to_data_at_location_all_usable_files(cfg)
good_location_df_fallen = good_location_df_fallen.reset_index()
good_location_df_fallen

Searching for files from Fallen Tree in 2022
Will save csv file to /Users/adityakrishna/Documents/bd2_dets_20241226/output_dir/Fallen
All files from deployment session good!
Will be looking at 3270 files from Fallen Tree


,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-08-19 02:16:29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 02:16:29 19/08/2022 (UTC) by Audio...,Usable for detection,36.9C,4.3V,192000,AudioMoth 24F3190361CBE96A,806.001270833333
1,2022-08-19 02:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 02:30:00 19/08/2022 (UTC) by Audio...,Usable for detection,35.5C,4.3V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
2,2022-08-19 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 03:00:00 19/08/2022 (UTC) by Audio...,Usable for detection,32.9C,4.3V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
3,2022-08-19 03:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 03:30:00 19/08/2022 (UTC) by Audio...,Usable for detection,31.7C,4.2V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
4,2022-08-19 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 04:00:00 19/08/2022 (UTC) by Audio...,Usable for detection,30.2C,4.2V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3265,2022-10-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 21:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,16.6C,4.0V,192000,AudioMoth 249BC30461CBE95C,300.001270833333
3266,2022-10-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 22:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,16.3C,4.0V,192000,AudioMoth 249BC30461CBE95C,300.001270833333
3267,2022-10-31 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 22:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,15.5C,4.0V,192000,AudioMoth 249BC30461CBE95C,300.001270833333
3268,2022-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 23:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,15.2C,4.0V,192000,AudioMoth 249BC30461CBE95C,300.001270833333


In [27]:
good_location_df_fallen_continuous = good_location_df_fallen[good_location_df_fallen['datetime_UTC']<=dt.datetime(2022,10,17,17,30,0)]
all_file_durations[cfg["site"].split()[0]] = good_location_df_fallen_continuous.set_index('datetime_UTC')['file_duration']
good_location_df_fallen[good_location_df_fallen['datetime_UTC']<=dt.datetime(2022,10,17,18,30,0)]

,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-08-19 02:16:29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 02:16:29 19/08/2022 (UTC) by Audio...,Usable for detection,36.9C,4.3V,192000,AudioMoth 24F3190361CBE96A,806.001270833333
1,2022-08-19 02:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 02:30:00 19/08/2022 (UTC) by Audio...,Usable for detection,35.5C,4.3V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
2,2022-08-19 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 03:00:00 19/08/2022 (UTC) by Audio...,Usable for detection,32.9C,4.3V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
3,2022-08-19 03:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 03:30:00 19/08/2022 (UTC) by Audio...,Usable for detection,31.7C,4.2V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
4,2022-08-19 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,003,/mnt/ubna_data_01_mir/recover-20220822/UBNA_00...,Recorded at 04:00:00 19/08/2022 (UTC) by Audio...,Usable for detection,30.2C,4.2V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2583,2022-10-17 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 16:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,21.5C,3.6V,192000,AudioMoth 249BC30461CBE95C,1795.00127083333
2584,2022-10-17 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 17:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,24.8C,3.6V,192000,AudioMoth 249BC30461CBE95C,1795.00127083333
2585,2022-10-17 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,011,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 17:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,27.2C,3.6V,192000,AudioMoth 249BC30461CBE95C,1153.96266666667
2586,2022-10-17 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,E,005,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 18:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,28.7C,4.2V,192000,AudioMoth 249BC30461CBE95C,300.001270833333


In [28]:
cfg = dict()
cfg["site"] = 'Opposite Central Pond'
cfg["year"] = '2022'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '23:59'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df_opposite, data_params = get_params_relevant_to_data_at_location_all_usable_files(cfg)
good_location_df_opposite = good_location_df_opposite.reset_index()
good_location_df_opposite

Searching for files from Opposite Central Pond in 2022
Will save csv file to /Users/adityakrishna/Documents/bd2_dets_20241226/output_dir/Opposite
Error files exist!
Will be looking at 2884 files from Opposite Central Pond


,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-08-23 01:33:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 01:33:50 23/08/2022 (UTC) by Audio...,Usable for detection,35.5C,4.2V,192000,AudioMoth 249BC30461CBEB1A,1565.00127083333
1,2022-08-23 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 02:00:00 23/08/2022 (UTC) by Audio...,Usable for detection,33.7C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
2,2022-08-23 02:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 02:30:00 23/08/2022 (UTC) by Audio...,Usable for detection,32.0C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
3,2022-08-23 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 03:00:00 23/08/2022 (UTC) by Audio...,Usable for detection,29.9C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
4,2022-08-23 03:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 03:30:00 23/08/2022 (UTC) by Audio...,Usable for detection,28.7C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2879,2022-10-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 21:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.0C,3.9V,192000,AudioMoth 249BC30461CBEB1A,300.001270833333
2880,2022-10-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 22:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,13.7C,3.9V,192000,AudioMoth 249BC30461CBEB1A,300.001270833333
2881,2022-10-31 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 22:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,13.4C,3.9V,192000,AudioMoth 249BC30461CBEB1A,300.001270833333
2882,2022-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 23:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,13.4C,3.9V,192000,AudioMoth 249BC30461CBEB1A,300.001270833333


In [29]:
good_location_df_opposite_continuous = good_location_df_opposite[good_location_df_opposite['datetime_UTC']<=dt.datetime(2022,10,17,17,30,0)]
all_file_durations[cfg["site"].split()[0]] = good_location_df_opposite_continuous.set_index('datetime_UTC')['file_duration']
good_location_df_opposite[good_location_df_opposite['datetime_UTC']<=dt.datetime(2022,10,17,18,30,0)]

,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-08-23 01:33:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 01:33:50 23/08/2022 (UTC) by Audio...,Usable for detection,35.5C,4.2V,192000,AudioMoth 249BC30461CBEB1A,1565.00127083333
1,2022-08-23 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 02:00:00 23/08/2022 (UTC) by Audio...,Usable for detection,33.7C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
2,2022-08-23 02:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 02:30:00 23/08/2022 (UTC) by Audio...,Usable for detection,32.0C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
3,2022-08-23 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 03:00:00 23/08/2022 (UTC) by Audio...,Usable for detection,29.9C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
4,2022-08-23 03:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,005,/mnt/ubna_data_01_mir/recover-20220825/UBNA_00...,Recorded at 03:30:00 23/08/2022 (UTC) by Audio...,Usable for detection,28.7C,4.1V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2197,2022-10-17 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 16:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,21.5C,3.5V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
2198,2022-10-17 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 17:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,30.7C,3.5V,192000,AudioMoth 249BC30461CBEB1A,1795.00127083333
2199,2022-10-17 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,012,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 17:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,36.9C,3.6V,192000,AudioMoth 249BC30461CBEB1A,1706.32533333333
2200,2022-10-17 18:01:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,006,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 18:01:39 17/10/2022 (UTC) by Audio...,Usable for detection,33.1C,4.1V,192000,AudioMoth 249BC30461CBEB1A,201.001270833333


In [30]:
cfg = dict()
cfg["site"] = 'Carp Pond'
cfg["year"] = '2022'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '23:59'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df_carp, data_params = get_params_relevant_to_data_at_location_all_usable_files(cfg)
good_location_df_carp = good_location_df_carp.reset_index()
good_location_df_carp

Searching for files from Carp Pond in 2022
Will save csv file to /Users/adityakrishna/Documents/bd2_dets_20241226/output_dir/Carp
Error files exist!
Will be looking at 5192 files from Carp Pond


,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-07-12 18:49:37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 18:49:37 12/07/2022 (UTC) by Audio...,Usable for detection,43.6C,4.3V,250000,AudioMoth 24F3190361CBE990,618.000976
1,2022-07-12 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 19:00:00 12/07/2022 (UTC) by Audio...,Usable for detection,37.5C,4.3V,250000,AudioMoth 24F3190361CBE990,1795.000976
2,2022-07-12 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 19:30:00 12/07/2022 (UTC) by Audio...,Usable for detection,33.6C,4.3V,250000,AudioMoth 24F3190361CBE990,1795.000976
3,2022-07-12 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 20:00:00 12/07/2022 (UTC) by Audio...,Usable for detection,32.6C,4.2V,250000,AudioMoth 24F3190361CBE990,1795.000976
4,2022-07-12 20:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 20:30:00 12/07/2022 (UTC) by Audio...,Usable for detection,33.3C,4.2V,250000,AudioMoth 24F3190361CBE990,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5187,2022-10-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 21:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.7C,4.0V,192000,AudioMoth 249BC30461CBE637,300.001270833333
5188,2022-10-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 22:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.5C,4.0V,192000,AudioMoth 249BC30461CBE637,300.001270833333
5189,2022-10-31 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 22:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.4C,4.0V,192000,AudioMoth 249BC30461CBE637,300.001270833333
5190,2022-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221110/UBNA_01...,Recorded at 23:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,13.7C,4.0V,192000,AudioMoth 249BC30461CBE637,300.001270833333


In [32]:
good_location_df_carp_continuous = good_location_df_carp[good_location_df_carp['datetime_UTC']<=dt.datetime(2022,10,17,17,30,0)]
all_file_durations[cfg["site"].split()[0]] = good_location_df_carp_continuous.set_index('datetime_UTC')['file_duration']
good_location_df_carp[good_location_df_carp['datetime_UTC']<=dt.datetime(2022,10,17,18,30,0)]

,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-07-12 18:49:37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 18:49:37 12/07/2022 (UTC) by Audio...,Usable for detection,43.6C,4.3V,250000,AudioMoth 24F3190361CBE990,618.000976
1,2022-07-12 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 19:00:00 12/07/2022 (UTC) by Audio...,Usable for detection,37.5C,4.3V,250000,AudioMoth 24F3190361CBE990,1795.000976
2,2022-07-12 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 19:30:00 12/07/2022 (UTC) by Audio...,Usable for detection,33.6C,4.3V,250000,AudioMoth 24F3190361CBE990,1795.000976
3,2022-07-12 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 20:00:00 12/07/2022 (UTC) by Audio...,Usable for detection,32.6C,4.2V,250000,AudioMoth 24F3190361CBE990,1795.000976
4,2022-07-12 20:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,008,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Recorded at 20:30:00 12/07/2022 (UTC) by Audio...,Usable for detection,33.3C,4.2V,250000,AudioMoth 24F3190361CBE990,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4505,2022-10-17 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 16:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,22.8C,3.8V,192000,AudioMoth 249BC30461CBE637,1795.00127083333
4506,2022-10-17 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 17:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,23.9C,3.8V,192000,AudioMoth 249BC30461CBE637,1795.00127083333
4507,2022-10-17 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,010,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Recorded at 17:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,25.0C,3.9V,192000,AudioMoth 249BC30461CBE637,440.234666666667
4508,2022-10-17 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,004,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 18:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,25.6C,4.2V,192000,AudioMoth 249BC30461CBE637,300.001270833333


In [33]:
cfg = dict()
cfg["site"] = 'Central Pond'
cfg["year"] = '2022'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '23:59'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df_central, data_params = get_params_relevant_to_data_at_location_all_usable_files(cfg)
good_location_df_central = good_location_df_central.reset_index()
good_location_df_central

Searching for files from Central Pond in 2022
Will save csv file to /Users/adityakrishna/Documents/bd2_dets_20241226/output_dir/Central
Error files exist!
Will be looking at 4300 files from Central Pond


,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-07-25 22:53:12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 22:53:12 25/07/2022 (UTC) by Audio...,Usable for detection,33.3C,4.3V,250000,AudioMoth 249BC30461CBE637,1.376256
1,2022-07-26 00:25:12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 00:25:12 26/07/2022 (UTC) by Audio...,Usable for detection,36.3C,4.3V,250000,AudioMoth 249BC30461CBE637,283.000976
2,2022-07-26 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 00:30:00 26/07/2022 (UTC) by Audio...,Usable for detection,38.5C,4.3V,250000,AudioMoth 249BC30461CBE637,1795.000976
3,2022-07-26 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 01:00:00 26/07/2022 (UTC) by Audio...,Usable for detection,39.3C,4.3V,250000,AudioMoth 249BC30461CBE637,1795.000976
4,2022-07-26 01:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 01:30:00 26/07/2022 (UTC) by Audio...,Usable for detection,42.9C,4.3V,250000,AudioMoth 249BC30461CBE637,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4295,2022-10-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,009,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 21:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,9.0C,4.0V,192000,AudioMoth 24F319055FDF2F5B,300.001270833333
4296,2022-10-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,009,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 22:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,8.7C,4.0V,192000,AudioMoth 24F319055FDF2F5B,300.001270833333
4297,2022-10-31 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,009,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 22:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,8.2C,4.0V,192000,AudioMoth 24F319055FDF2F5B,300.001270833333
4298,2022-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,009,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 23:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,8.2C,4.0V,192000,AudioMoth 24F319055FDF2F5B,300.001270833333


In [34]:
good_location_df_central_continuous = good_location_df_central[good_location_df_central['datetime_UTC']<=dt.datetime(2022,10,17,17,30,0)]
all_file_durations[cfg["site"].split()[0]] = good_location_df_central_continuous.set_index('datetime_UTC')['file_duration']
good_location_df_central[good_location_df_central['datetime_UTC']<=dt.datetime(2022,10,17,18,30,0)]

,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-07-25 22:53:12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 22:53:12 25/07/2022 (UTC) by Audio...,Usable for detection,33.3C,4.3V,250000,AudioMoth 249BC30461CBE637,1.376256
1,2022-07-26 00:25:12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 00:25:12 26/07/2022 (UTC) by Audio...,Usable for detection,36.3C,4.3V,250000,AudioMoth 249BC30461CBE637,283.000976
2,2022-07-26 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 00:30:00 26/07/2022 (UTC) by Audio...,Usable for detection,38.5C,4.3V,250000,AudioMoth 249BC30461CBE637,1795.000976
3,2022-07-26 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 01:00:00 26/07/2022 (UTC) by Audio...,Usable for detection,39.3C,4.3V,250000,AudioMoth 249BC30461CBE637,1795.000976
4,2022-07-26 01:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D,008,/mnt/ubna_data_01_mir/recover-20220728/UBNA_00...,Recorded at 01:30:00 26/07/2022 (UTC) by Audio...,Usable for detection,42.9C,4.3V,250000,AudioMoth 249BC30461CBE637,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3971,2022-10-17 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,009,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 16:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,15.8C,3.8V,192000,AudioMoth 24F319055FDF2F5B,1795.00127083333
3972,2022-10-17 17:30:28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,003,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 17:30:28 17/10/2022 (UTC) by Audio...,Usable for detection,28.0C,4.2V,192000,AudioMoth 24F319055FDF2F5B,5.97333333333333
3973,2022-10-17 17:30:43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,003,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 17:30:43 17/10/2022 (UTC) by Audio...,Usable for detection,29.0C,4.2V,192000,AudioMoth 24F319055FDF2F5B,257.001270833333
3974,2022-10-17 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,003,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 18:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,18.7C,4.2V,192000,AudioMoth 24F319055FDF2F5B,300.001270833333


In [35]:
cfg = dict()
cfg["site"] = 'Foliage'
cfg["year"] = '2022'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '23:59'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df_foliage, data_params = get_params_relevant_to_data_at_location_all_usable_files(cfg)
good_location_df_foliage = good_location_df_foliage.reset_index()
good_location_df_foliage

Searching for files from Foliage in 2022
Will save csv file to /Users/adityakrishna/Documents/bd2_dets_20241226/output_dir/Foliage
Error files exist!
Will be looking at 5932 files from Foliage


,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-06-14 22:36:27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 22:36:27 14/06/2022 (UTC) by Audio...,Usable for detection,27.5C,4.7V,250000,AudioMoth 24F319055FDF2F5B,1408.000976
1,2022-06-14 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 23:00:00 14/06/2022 (UTC) by Audio...,Usable for detection,18.8C,4.7V,250000,AudioMoth 24F319055FDF2F5B,1795.000976
2,2022-06-14 23:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 23:30:00 14/06/2022 (UTC) by Audio...,Usable for detection,17.7C,4.6V,250000,AudioMoth 24F319055FDF2F5B,1795.000976
3,2022-06-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 00:00:00 15/06/2022 (UTC) by Audio...,Usable for detection,16.7C,4.6V,250000,AudioMoth 24F319055FDF2F5B,421.527552
4,2022-06-15 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 00:30:00 15/06/2022 (UTC) by Audio...,Usable for detection,16.0C,4.5V,250000,AudioMoth 24F319055FDF2F5B,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5927,2022-10-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 21:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,16.0C,4.0V,192000,AudioMoth 24F3190361CBE96A,300.001270833333
5928,2022-10-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 22:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,15.3C,4.0V,192000,AudioMoth 24F3190361CBE96A,300.001270833333
5929,2022-10-31 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 22:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.4C,4.0V,192000,AudioMoth 24F3190361CBE96A,300.001270833333
5930,2022-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 23:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.5C,4.0V,192000,AudioMoth 24F3190361CBE96A,300.001270833333


In [36]:
good_location_df_foliage_continuous = good_location_df_foliage[good_location_df_foliage['datetime_UTC']<=dt.datetime(2022,10,17,17,0,0)]
all_file_durations[cfg["site"].split()[0]] = good_location_df_foliage_continuous.set_index('datetime_UTC')['file_duration']
good_location_df_foliage[good_location_df_foliage['datetime_UTC']<=dt.datetime(2022,10,17,17,30,0)]

,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-06-14 22:36:27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 22:36:27 14/06/2022 (UTC) by Audio...,Usable for detection,27.5C,4.7V,250000,AudioMoth 24F319055FDF2F5B,1408.000976
1,2022-06-14 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 23:00:00 14/06/2022 (UTC) by Audio...,Usable for detection,18.8C,4.7V,250000,AudioMoth 24F319055FDF2F5B,1795.000976
2,2022-06-14 23:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 23:30:00 14/06/2022 (UTC) by Audio...,Usable for detection,17.7C,4.6V,250000,AudioMoth 24F319055FDF2F5B,1795.000976
3,2022-06-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 00:00:00 15/06/2022 (UTC) by Audio...,Usable for detection,16.7C,4.6V,250000,AudioMoth 24F319055FDF2F5B,421.527552
4,2022-06-15 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C,010,/mnt/ubna_data_01_mir/recover-20220616_unit2/2...,Recorded at 00:30:00 15/06/2022 (UTC) by Audio...,Usable for detection,16.0C,4.5V,250000,AudioMoth 24F319055FDF2F5B,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5474,2022-10-17 15:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 15:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,17.9C,3.6V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
5475,2022-10-17 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 16:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,19.9C,3.6V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
5476,2022-10-17 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 16:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,21.2C,3.6V,192000,AudioMoth 24F3190361CBE96A,1795.00127083333
5477,2022-10-17 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,B,008,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 17:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,22.8C,3.6V,192000,AudioMoth 24F3190361CBE96A,1203.37066666667


In [37]:
cfg = dict()
cfg["site"] = 'Telephone Field'
cfg["year"] = '2022'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '23:59'
cfg['duration'] = 1795
cfg["output_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output_dir')
cfg["tmp_dir"] = Path(f'{Path.home()}/Documents/bd2_dets_20241226/output')
cfg["skip_existing"] = False
cfg["num_processes"] = multiprocessing.cpu_count()

good_location_df_telephone, data_params = get_params_relevant_to_data_at_location_all_usable_files(cfg)
good_location_df_telephone = good_location_df_telephone.reset_index()
good_location_df_telephone

Searching for files from Telephone Field in 2022
Will save csv file to /Users/adityakrishna/Documents/bd2_dets_20241226/output_dir/Telephone
All files from deployment session good!
Will be looking at 3416 files from Telephone Field


,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-07-22 21:08:42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 21:08:42 22/07/2022 (UTC) by Audio...,Usable for detection,31.5C,4.3V,250000,AudioMoth 249BC30461CBEB1A,1273.000976
1,2022-07-22 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 21:30:00 22/07/2022 (UTC) by Audio...,Usable for detection,28.5C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
2,2022-07-22 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 22:00:00 22/07/2022 (UTC) by Audio...,Usable for detection,27.9C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
3,2022-07-22 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 22:30:00 22/07/2022 (UTC) by Audio...,Usable for detection,28.8C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
4,2022-07-22 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 23:00:00 22/07/2022 (UTC) by Audio...,Usable for detection,28.8C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3411,2022-10-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 21:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.4C,4.0V,192000,AudioMoth 24F3190361CBE990,300.001270833333
3412,2022-10-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 22:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,14.0C,4.0V,192000,AudioMoth 24F3190361CBE990,300.001270833333
3413,2022-10-31 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 22:30:00 31/10/2022 (UTC) by Audio...,Usable for detection,13.7C,4.0V,192000,AudioMoth 24F3190361CBE990,300.001270833333
3414,2022-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221110/UBNA_00...,Recorded at 23:00:00 31/10/2022 (UTC) by Audio...,Usable for detection,13.4C,4.0V,192000,AudioMoth 24F3190361CBE990,300.001270833333


In [38]:
good_location_df_telephone_continuous = good_location_df_telephone[good_location_df_telephone['datetime_UTC']<=dt.datetime(2022,10,17,17,0,0)]
all_file_durations[cfg["site"].split()[0]] = good_location_df_telephone_continuous.set_index('datetime_UTC')['file_duration']
good_location_df_telephone[good_location_df_telephone['datetime_UTC']<=dt.datetime(2022,10,17,17,30,0)]

,datetime_UTC,Datetime UTC,Site name,Recover folder,AudioMoth #,SD card #,File path,File metadata,File status,Audiomoth temperature,...,audiomoth_num,sd_card_num,file_path,file_metadata,file_status,audiomoth_temperature,audiomoth_battery,sample_rate,audiomoth_artist_ID,file_duration
0,2022-07-22 21:08:42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 21:08:42 22/07/2022 (UTC) by Audio...,Usable for detection,31.5C,4.3V,250000,AudioMoth 249BC30461CBEB1A,1273.000976
1,2022-07-22 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 21:30:00 22/07/2022 (UTC) by Audio...,Usable for detection,28.5C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
2,2022-07-22 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 22:00:00 22/07/2022 (UTC) by Audio...,Usable for detection,27.9C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
3,2022-07-22 22:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 22:30:00 22/07/2022 (UTC) by Audio...,Usable for detection,28.8C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
4,2022-07-22 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,F,010,/mnt/ubna_data_01_mir/recover-20220725/UBNA_01...,Recorded at 23:00:00 22/07/2022 (UTC) by Audio...,Usable for detection,28.8C,4.2V,250000,AudioMoth 249BC30461CBEB1A,1795.000976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2022-10-17 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 16:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,19.3C,3.7V,192000,AudioMoth 24F3190361CBE990,1795.00127083333
2728,2022-10-17 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 16:30:00 17/10/2022 (UTC) by Audio...,Usable for detection,20.6C,3.7V,192000,AudioMoth 24F3190361CBE990,1795.00127083333
2729,2022-10-17 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,007,/mnt/ubna_data_02_mir/recover-20221017/UBNA_00...,Recorded at 17:00:00 17/10/2022 (UTC) by Audio...,Usable for detection,21.7C,3.7V,192000,AudioMoth 24F3190361CBE990,229.973333333333
2730,2022-10-17 17:09:55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,A,001,/mnt/ubna_data_02_mir/recover-20221027/UBNA_00...,Recorded at 17:09:55 17/10/2022 (UTC) by Audio...,Usable for detection,29.0C,4.2V,192000,AudioMoth 24F3190361CBE990,2.47466666666667


In [39]:
all_file_durations

{'Fallen': datetime_UTC
 2022-08-19 02:16:29    806.001270833333
 2022-08-19 02:30:00    1795.00127083333
 2022-08-19 03:00:00    1795.00127083333
 2022-08-19 03:30:00    1795.00127083333
 2022-08-19 04:00:00    1795.00127083333
                              ...       
 2022-10-17 15:30:00    1795.00127083333
 2022-10-17 16:00:00    1795.00127083333
 2022-10-17 16:30:00    1795.00127083333
 2022-10-17 17:00:00    1795.00127083333
 2022-10-17 17:30:00    1153.96266666667
 Name: file_duration, Length: 2586, dtype: object,
 'Opposite': datetime_UTC
 2022-08-23 01:33:50    1565.00127083333
 2022-08-23 02:00:00    1795.00127083333
 2022-08-23 02:30:00    1795.00127083333
 2022-08-23 03:00:00    1795.00127083333
 2022-08-23 03:30:00    1795.00127083333
                              ...       
 2022-10-17 15:30:00    1795.00127083333
 2022-10-17 16:00:00    1795.00127083333
 2022-10-17 16:30:00    1795.00127083333
 2022-10-17 17:00:00    1795.00127083333
 2022-10-17 17:30:00    1706.325333333

In [52]:
amount_of_time_from_site = dict()
for key in all_file_durations.keys():
    file_durations = all_file_durations[key]
    amount_of_time_from_site[key] = (file_durations.astype(float).sum() / 60) / 60

In [54]:
amount_of_time_from_site

{'Fallen': 1284.141381556711,
 'Opposite': 1090.2254390162018,
 'Carp': 2230.111366184256,
 'Central': 1968.1174082497191,
 'Foliage': 2705.8355698987934,
 'Telephone': 1352.6101900371273}

In [58]:
total_number_of_hours_from_all_sites = pd.DataFrame([amount_of_time_from_site]).sum(axis=1).item()
total_number_of_hours_from_all_sites

10631.041354942809